# ALL Detection - Kaggle Training Quickstart

This notebook allows you to train the ALL Detection model using Kaggle's P100/T4 GPUs. 

### ⚠️ Setup Required:
1. **Add Dataset**: Click "+ Add Data" in the right sidebar. Search for "C-NMC leukemia" and add the official dataset.
2. **GPU**: In the right sidebar under "Settings", ensure "Accelerator" is set to **GPU P100** or **GPU T4 x2**.

## 1. Clone Repository & Install Dependencies

In [ ]:
import os
%cd /kaggle/working
!rm -rf ALL-Detection
!git clone https://github.com/Rujul-Rumale/ALL-Detection.git
%cd /kaggle/working/ALL-Detection
!pip install -r requirements.txt
!pip install kornia  # Ensure kornia is installed

## 2. Rebuild Data Splits
Since we are on Linux (Kaggle), we need to regenerate the split JSON so the training script can find the images.

In [ ]:
%cd /kaggle/working/ALL-Detection
!python training_scripts/build_cv_splits.py

## 3. Begin Training!
Outputs will be saved in `/kaggle/working/ALL_Kaggle_Outputs`.

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/ALL_Kaggle_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── MNV3L (Student) ──────────────────────────────────────
for fold in [1, 2, 3]:
    print(f"\n>>> MNV3L Fold {fold}")
    ret = os.system(f"""
        cd /kaggle/working/ALL-Detection && \
        python training_scripts/train_colab.py \
          --model mnv3l \
          --fold {fold} \
          --run_name mnv3l_kaggle \
          --epochs 100 \
          --cosine_t_max 100 \
          --batch_size 128 \
          --num_workers 2 \
          --no_live \
          --output_root {OUTPUT_DIR}
    """)
    print(f"Fold {fold} exited with code {ret}")

# ── EffB4 (Teacher) — run in separate session ─────────────
# for fold in [1, 2, 3]:
#     print(f"\n>>> EffB4 Fold {fold}")
#     ret = os.system(f"""
#         cd /kaggle/working/ALL-Detection && \
#         python training_scripts/train_colab.py \
#           --model effb4 \
#           --fold {fold} \
#           --run_name effb4_kaggle \
#           --epochs 80 \
#           --cosine_t_max 80 \
#           --phase1_start 6 \
#           --phase1_5_start 11 \
#           --phase2_start 21 \
#           --batch_size 64 \
#           --num_workers 2 \
#           --no_live \
#           --output_root {OUTPUT_DIR}
#     """)
#     print(f"Fold {fold} exited with code {ret}")

# ── ResNet50 (Teacher) — run in separate session ──────────
# for fold in [1, 2, 3]:
#     print(f"\n>>> ResNet50 Fold {fold}")
#     ret = os.system(f"""
#         cd /kaggle/working/ALL-Detection && \
#         python training_scripts/train_colab.py \
#           --model rn50 \
#           --fold {fold} \
#           --run_name rn50_kaggle \
#           --epochs 80 \
#           --cosine_t_max 80 \
#           --phase1_start 6 \
#           --phase1_5_start 11 \
#           --phase2_start 21 \
#           --batch_size 96 \
#           --num_workers 2 \
#           --no_live \
#           --output_root {OUTPUT_DIR}
#     """)
#     print(f"Fold {fold} exited with code {ret}")


## 4. VRAM Check

In [ ]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total",
     "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
used, total = result.stdout.strip().split(", ")
print(f"VRAM: {int(used)/1024:.1f} / {int(total)/1024:.1f} GB")
print(f"Headroom: {(int(total)-int(used))/1024:.1f} GB")
print("If headroom < 1.5 GB after epoch 1, reduce --batch_size before continuing.")
